In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# The Performance Model

Every network we have trained so far spends most of its arithmetic in
matrix products, and we have assumed that a GPU computes them at the rate
its specification states. The assumption is often wrong. On the RTX 4090
that runs this book's notebooks, a product of two $8192 \times 8192$
matrices in bf16 comes close to the 165 TFLOP/s that the specification
promises. A product of two $256 \times 256$ matrices reaches less than two
percent of it, although the operation and the chip are the same and both
products are timed correctly. A training step is a long sequence of
operations, many of them small. If most of them run at a few percent of the
peak, a faster GPU or a second one changes little, because the time goes to
waiting rather than to computing. Why does the same operation run at such
different speeds, and how do we tell which case a program is in?

This section explains the gap. Every operation on a GPU is limited by one of
three resources: the arithmetic units, the memory bandwidth that supplies
them with data, or the fixed cost of issuing the operation from Python. A
single number computed from the shapes of an operation, its arithmetic
intensity, tells which of the first two applies. Only measurement reveals
the third. Together they give a method in four steps, which the rest of
this chapter follows. We **measure** what a program does, **classify** which
resource limits it, **fix** that resource with the matching technique, and
**re-measure**, because each fix moves the bottleneck somewhere else.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import re
import time

In [ ]:
# Run fp32 matrix products on the tf32 tensor cores (PyTorch's default is
# full fp32). The bf16 products below are unaffected.
torch.set_float32_matmul_precision('high')

## Arithmetic Intensity and the Roofline

Before we time anything, we can predict from the shapes of an operation
which resource should limit it. The prediction needs two counts: the
arithmetic the operation performs and the bytes it moves.

### Counting FLOPs and Bytes

Every tensor operation makes two demands on a GPU. It performs arithmetic,
which we count in *floating-point operations* (FLOPs), one addition or one
multiplication each. And it moves bytes: its inputs come from the GPU's
memory, and its outputs go back there. The GPU has a separate budget for
each demand. The RTX 4090 performs at most $P = 165 \times 10^{12}$ FLOP/s
in bf16 on its tensor cores, and its memory delivers at most
$\beta = 1.008 \times 10^{12}$ bytes/s. Which budget runs out first depends
on the ratio of the two demands, the *arithmetic intensity*

$$
I \;=\; \frac{\textrm{FLOPs performed}}{\textrm{bytes moved}}.
$$

An operation of high intensity performs many FLOPs on each byte it
fetches. An operation of low intensity performs only a few, or less than
one.

The most important operation in deep learning is the matrix product
$\mathbf{X}\mathbf{W}$ with $\mathbf{X} \in \mathbb{R}^{B \times D}$ and
$\mathbf{W} \in \mathbb{R}^{D \times F}$, a linear layer applied to a
minibatch of $B$ examples. It performs $2BDF$ FLOPs, one multiplication and
one addition for each of the $BDF$ terms of the sums. At best it reads
$\mathbf{X}$ and $\mathbf{W}$ once and writes the result once, which moves
$BD + DF + BF$ numbers. With $b$ bytes per number, its intensity is

$$
\frac{2BDF}{b\,(BD + DF + BF)}
\;=\; \frac{2}{b}\left(\frac{1}{F} + \frac{1}{B} + \frac{1}{D}\right)^{-1}
\;\approx\; \frac{2B}{b} \quad \textrm{for } B \ll D, F.
$$

The middle form shows that the smallest of the three dimensions limits the
intensity. The approximation covers the common case of a minibatch that is
narrower than the layer. There the intensity grows linearly with the batch
size, because every weight fetched from memory is used once for each row of
the batch. A batch of one uses each weight once and discards it.

A GPT with about a billion parameters has attention projections of width
$D = F = 2048$. In training, a minibatch of 8 sequences of 1,024 tokens
gives $B = 8192$, and the exact form of the equation gives
about 910 FLOP/byte in bf16 ($b = 2$). Here the width, not the batch, is the
smallest dimension. When the same model generates text one token at a time,
$B = 1$ and the intensity is about 1 FLOP/byte. The same layer, with the
same weights, spans a factor of about a thousand in intensity between the
two uses.

An elementwise operation lies at the low extreme. The operation
$y_i = x_i + 1$ performs one FLOP per element and moves $2b$ bytes, since it
reads $x_i$ and writes $y_i$. Its intensity is $1/(2b)$, or 0.25 FLOP/byte
in bf16, however large the tensor.

### The Roofline Model

The *roofline model* [@Williams.Waterman.Patterson.2009] turns
intensity into a bound on speed. An operation cannot run faster than the
arithmetic units allow, and it cannot run faster than memory delivers its
bytes, which caps it at $I \beta$ FLOP/s. The attainable throughput is
therefore

$$
\textrm{performance} \;\leq\; \min(P,\; I \beta).
$$

On log–log axes the bound is a sloped line that meets a flat roof
(the figure). The two meet at the *ridge point*
$I^* = P/\beta$. An operation of intensity below $I^*$ is
*bandwidth-bound*: memory cannot deliver bytes fast enough to keep the
arithmetic units busy, so additional arithmetic on each byte costs no time.
An operation above $I^*$ is *compute-bound*: the arithmetic units are
saturated, and moving fewer bytes would not make it faster.
The elementwise operation $x + 1$ in bf16 has an intensity of
0.25 FLOP/byte, so it runs at most at $0.25\beta$, about a quarter of a
trillion FLOP/s. That is less than a six-hundredth of the compute roof.

![The roofline of the RTX 4090. Attainable throughput is the smaller of the compute roof $P$ and the arithmetic intensity times the memory bandwidth $\beta$. The points mark the examples of the text: $x + 1$ in bf16, a linear layer of width 2048 applied to one token, and the same layer applied to a minibatch of 8,192 tokens. The ridge point lies near 164 FLOP/byte.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-roofline.svg)

### The Ridge Point of the RTX 4090

The peak throughput of the tensor cores cannot be queried from the device,
so we take $P$ and $\beta$ from the specification sheet and print the name
of the GPU that the code runs on. On another GPU, replace the two constants
with the values from its specification.

In [ ]:
print(torch.cuda.get_device_name())

In [ ]:
peak_tflops = 165.2    # RTX 4090: dense bf16 on the tensor cores
bandwidth_tbs = 1.008  # RTX 4090: GDDR6X memory
print(f'ridge point = {peak_tflops / bandwidth_tbs:.0f} FLOP/byte')

A ridge point of 164 FLOP/byte means that an operation must perform about
164 FLOPs for every byte it moves before the arithmetic units, rather than
memory, set its speed. On other current GPUs the ridge lies at about
120 FLOP/byte (RTX 5090) or about 300 (H100, B200). Only large matrix
products reach these values. Elementwise operations, normalizations, and
the generation of text one token at a time all lie far below the ridge,
and most of this chapter is therefore about moving fewer bytes.

For a square product with $B = D = F = n$ in bf16 the intensity is $n/3$,
so the roofline model declares every such product with $n$ above about 500
compute-bound. To see whether the GPU agrees, we first need a way to
measure it.

## Timing GPU Code

Python does not wait for the GPU. As that section showed,
each operation is placed in the device's queue and Python moves on: 32
products of $4096 \times 4096$ matrices are issued in a small fraction of
the time the GPU needs to compute them. A clock read right after a call
therefore measures how long Python took to queue the work, not how long the
GPU took to do it (the figure). A correct measurement
runs the operation once before timing it, waits for the device before it
starts and before it stops the clock, and divides the time of many
repetitions by their number.

![Asynchronous dispatch. Python enqueues kernels and moves on, and the GPU
executes them behind it. A timer read without a barrier stops when the last
kernel has been *enqueued*; only a synchronization point waits until the
work is *done*.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-async-timeline.svg)

### Synchronization Points

Python waits for the device only when it needs a value that the device
computes: to convert it into a Python number or a NumPy array, to print it,
or to branch on it, as in `if loss < 0.1:`. Each such *synchronization
point* drains the queue up to that value. One kind is easy to miss: an
operation whose output *shape* depends on the data, such as selecting the
positive entries `x[x > 0]`, waits because Python must learn the shape of
the result before it can continue.

After a synchronization point the device sits idle until Python sends more
work. A training loop that reads its loss on every step therefore drains
the queue once per step. To see what this costs, we time two loops of 1,000
steps. Each step multiplies a $1024 \times 1024$ matrix by itself and adds
the sum of the result to a running total on the device. This work takes
the GPU some tens of microseconds, comparable to the time Python needs to
issue it. The two loops differ only in when they read the total: the first
converts it into a Python number after every step, the second once at the
end.

In [ ]:
x = torch.ones(1024, 1024, device=d2l.try_gpu())
s = torch.zeros((), device=x.device)
s += (x @ x).sum()
float(s)                          # Warm up: load the kernels

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):
    s += (x @ x).sum()
    total = float(s)              # Reads the value: a barrier every step
read_every = time.perf_counter() - t0

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):
    s += (x @ x).sum()            # Stays on the device: no barrier
total = float(s)                  # One read at the end
read_once = time.perf_counter() - t0
print(f'read every step: {read_every:.3f} s   read once: {read_once:.3f} s')

`torch.cuda.synchronize()` waits until the device has finished all queued
work, so that each loop starts on an idle device. `float(s)` converts a
tensor with one element into a Python number, which waits for its value.

Reading the value on every step makes the loop slower, although the
device does the same work. Without reads, host and device work at the same
time: while the GPU computes one step, Python issues the next, and each
step costs the longer of the two times. A read on every step makes them
take turns. The device idles while the host waits for the value and then
issues the next step, and the host idles while the device computes. Each
step then costs the host time plus the device time plus the latency of the
round trip.

Here the reads make the loop about one and a half times as slow.

A training loop should therefore read values from the device once per
logging interval, not once per step, and keep running sums on the device.
The `d2l.Trainer` follows this rule: its training and validation steps return
losses as device arrays, which it adds up on the device and converts into
Python numbers once per epoch.

### A Benchmark Helper

Every timing in this chapter follows the same rules, which `d2l.Benchmark`
of that section packages. It calls a function a few times
to warm up, which excludes one-time costs such as loading kernels, selecting
algorithms, and compiling. It then waits for the device, times `repeats`
further calls, waits again, and stores the average time per call.

On a machine without a GPU, PyTorch runs operations synchronously, and
`Benchmark` needs no barrier.

We time the product of two $4096 \times 4096$ matrices in bf16.

In [ ]:
a = torch.randn(4096, 4096, device=d2l.try_gpu(), dtype=torch.bfloat16)
d2l.Benchmark(lambda: torch.mm(a, a), desc='4096 x 4096 product')

Each product takes a little under a millisecond. It performs
$2 \cdot 4096^3 \approx 1.4 \times 10^{11}$ FLOPs, so the GPU computes at
more than 150 TFLOP/s, over 90% of its specification.

## Three Regimes

The time of a GPU program divides among compute-bound kernels,
bandwidth-bound kernels, and idle gaps in which the device waits for Python
to send work [@He.2022]. Whichever part dominates names the
program's *regime*. Two experiments locate the three regimes on the
RTX 4090: matrix products of growing size, and elementwise operations on a
large tensor.

### Matrix Products across Sizes

Does a square product reach the roof as soon as the roofline says it
should, near $n = 500$? We time products in bf16 from $n = 256$ to
$n = 8192$. Each size runs a hundred times, so that the latency of the final
wait is spread over many calls.

In [ ]:
sizes = [256, 512, 1024, 2048, 4096, 8192]
times = []
for n in sizes:
    a = torch.randn(n, n, device=d2l.try_gpu(), dtype=torch.bfloat16)
    times.append(d2l.Benchmark(lambda: torch.mm(a, a), repeats=100).time)

We convert each time into TFLOP/s and plot it against the intensity $n/3$,
together with the roofline bound the equation, which has its corner
at the ridge point.

In [ ]:
measured = [2 * n**3 / t / 1e12 for n, t in zip(sizes, times)]
intensity = [n / 3 for n in sizes]  # FLOP/byte of a square bf16 product
ridge = peak_tflops / bandwidth_tbs
roof_x = [intensity[0], ridge, intensity[-1]]
roof_y = [min(peak_tflops, bandwidth_tbs * i) for i in roof_x]
d2l.plot([roof_x, intensity], [roof_y, measured],
         'arithmetic intensity (FLOP/byte)', 'TFLOP/s',
         legend=['roofline bound', 'measured'],
         xscale='log', yscale='log', fmts=('-', 'o'))

Printing the time per call of each size shows where the measurements fall
short of the bound.

In [ ]:
for n, t, tf in zip(sizes, times, measured):
    print(f'n = {n:4d}: {1e6 * t:7.1f} us per call, {tf:5.1f} TFLOP/s')

The roofline bound is flat from $n = 512$ on, but the measurements reach it
only at the largest sizes. From $n = 2048$ on, the GPU computes within about
15% of its specification: these products are compute-bound in fact as well
as in the model. A measurement slightly above the specification is
possible, because the specification assumes the card's rated clock of
2.52 GHz and the card often runs faster.

The small sizes behave differently. From $n = 256$ to $n = 1024$ the work
grows 64-fold, but the time per call changes by less than a factor of two,
and each doubling of $n$ multiplies the throughput by five to ten. A time
per call that barely depends on the size cannot be spent on arithmetic. It is the time $h$ that the host needs to issue one call:
Python, the framework, and the driver that launches the kernel. When calls
are queued back to back, host and device work at the same time, and each
call takes the longer of the two:

$$
t_\textrm{call} \;\approx\; \max\left(h,\; \frac{2n^3}{P}\right).
$$

For small $n$ the device finishes each product before the host has issued
the next one, and the throughput $2n^3/h$ rises with the cube of $n$. This
is the third regime, *overhead-bound*. The knee of the curve lies where the
device time of a product overtakes the host time,
$n \approx (hP/2)^{1/3}$: for $h = 20\,\mu\textrm{s}$ and the peak of the
RTX 4090, near $n = 1200$. The time per call at the smallest sizes in the
output above is the value of $h$ for this framework and this machine.

The roofline has no term for overhead. It is a ceiling, and the measured
points meet it only for operations large enough that the host time per call
is negligible. The faster the GPU, the larger an operation must be to get
there: $n \approx (hP/2)^{1/3}$ grows with $P$.

A product can also fall short of the roof for a reason inside the device.
A GPU executes a kernel on its *streaming multiprocessors* (SMs), the
independent processors of the chip; the RTX 4090 has 128 of them
(that section). A matrix-product kernel splits the output into
tiles and gives each SM one tile at a time. A product whose output has
fewer tiles than the GPU has SMs leaves some SMs idle, however fast the
host issues it. The share of the peak that it can reach is then at most
the share of SMs that hold a tile.

### Elementwise Operations

Products large enough to escape the overhead regime lie above the ridge
point, so the sweep never shows the bandwidth-bound regime. To see it, we
apply single elementwise operations of very different arithmetic cost to a
tensor of $8192 \times 8192$ numbers in fp32, 256 MiB: an addition, a
multiplication, a sine, and a sigmoid. Each reads the tensor once and
writes a result of the same size.

In [ ]:
x = torch.randn(8192, 8192, device=d2l.try_gpu())  # 256 MiB in fp32
for desc, f in [('add', lambda x: x + 1), ('mul', lambda x: x * 1.5),
                ('sin', torch.sin), ('sigmoid', torch.sigmoid)]:
    print(d2l.Benchmark(lambda: f(x), desc=desc))

All four operations take the same time, a little over half a millisecond.
Each moves 512 MiB, $5.4 \times 10^8$ bytes, which amounts to about
0.9 TB/s, close to the 1.008 TB/s of the specification. A sine costs many
more FLOPs than an addition, yet it runs no slower, because the bytes each
kernel moves set its time. This is the defining behavior of a
bandwidth-bound operation: additional arithmetic costs no time until the
intensity approaches the ridge point.

Activation functions and normalization layers chain several elementwise
operations. The function below is the tanh approximation of the GELU
activation $z\,\Phi(z)$ (that section) with
rounded constants, and it applies eight elementwise operations to its
input. Executed one operation at a time, each operation is a kernel of its
own, which reads its inputs from memory and writes its result back.

In [ ]:
def gelu_ish(x):
    """The tanh approximation of GELU with rounded constants: a chain of
    eight elementwise operations."""
    return 0.5 * x * (1 + torch.tanh(0.8 * (x + 0.04 * x**3)))

print(d2l.Benchmark(lambda: x + 1, desc='1 op'))
print(d2l.Benchmark(lambda: gelu_ish(x), desc='chain'))  # Eager: 8 kernels

The chain takes eight to nine times as long as the single addition, and
the bytes explain the factor. The addition reads one tensor and writes one. The
eight operations of the chain read and write 18 tensors of the same size
between them. Each of the six operations with one tensor input moves two;
the sum $x + 0.04x^3$ and the final product combine two tensors and move
three each. One kernel that read `x` once and wrote the result once would
move 2 tensors instead of 18. Merging operations into such a kernel is
called *fusion*, and a compiler can do it automatically.

### Diagnosis and Fixes

The experiments have located the three regimes on one GPU
(the figure). Each regime has a symptom by which we recognize it
and a standard fix, summarized in the table. The later
sections of this chapter develop each fix.

![The three regimes. Where the time of a step goes, among compute-bound
kernels, bandwidth-bound kernels, and idle gaps, determines which fix
applies.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-regimes.svg)

<!-- tbl-caption:Symptoms and standard fixes of the three regimes. -->

| regime | symptom | standard fix |
|---|---|---|
| compute-bound | device busy; intensity above the ridge; time grows with the FLOPs | a format with faster tensor cores; more devices |
| bandwidth-bound | device busy; intensity below the ridge; time grows with the bytes moved | fusion; recomputation instead of storage; algorithms that reuse data on chip |
| overhead-bound | device idle between kernels; time per call independent of size | fewer and larger calls; capturing the whole program and replaying it without Python |

The FlashAttention kernel of that section is an example
of the third fix for bandwidth: it computes attention in tiles held on the
chip, so the $T \times T$ matrix of attention scores never reaches memory. A
wrong diagnosis wastes effort. More FLOP/s do not speed up a bandwidth-bound
operation, and a faster GPU does not help a program whose device sits idle.

## Profiling a Training Step

A training step runs hundreds of kernels from all three regimes. A
*profiler* records every kernel the device runs, with its duration, so that
we can add up where the time of a step goes: how much each kernel takes,
and how long the device sits idle.

### Recording the Kernels

We profile one SGD step of an MLP with two hidden layers of width 4096 on
a minibatch of 1,024 examples, in fp32, with the matrix products on the
tf32 tensor cores. First we measure the
wall-clock time per step with `Benchmark`.

In [ ]:
net = nn.Sequential(nn.Linear(4096, 4096), nn.ReLU(),
                    nn.Linear(4096, 4096), nn.ReLU(),
                    nn.Linear(4096, 10)).to(d2l.try_gpu())
X = torch.randn(1024, 4096, device=d2l.try_gpu())
opt = torch.optim.SGD(net.parameters(), lr=0.01)

def step():
    opt.zero_grad()
    net(X).mean().backward()
    opt.step()

wall = d2l.Benchmark(step).time

`kernel_times` runs a function ten times under the profiler and returns
the device time per call of every kernel that the function launched. We
save it, and the printout below, for the case study at the end of the
chapter.

In [ ]:
def kernel_times(f, steps=10):
    """Device milliseconds per call of f, for every kernel that f runs."""
    with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CUDA],
            acc_events=True) as prof:
        for _ in range(steps):
            f()
        torch.cuda.synchronize()
    return {e.key: e.device_time_total / 1e3 / steps
            for e in prof.key_averages()
            if e.device_type == torch.autograd.DeviceType.CUDA}

`prof.key_averages()` groups the recorded events by name. Kernels are the
events that ran on the device; the other events, such as the calls that
launch kernels, ran on the CPU. `prof.export_chrome_trace('trace.json')`
writes the full timeline, which <https://ui.perfetto.dev> displays.

### Reading the Profile

`show_profile` prints the device time per step against the wall-clock
time, the share of the matrix products, and the kernels that take the most
device time. It strips the return type, the namespaces, and the argument
list from the C++ names of the kernels.

In [ ]:
def show_profile(kernels, step_ms, rows=6):
    """Busy device time against the wall-clock time step_ms of a step, the
    share of the matrix products, and the kernels with the most time."""
    busy = sum(kernels.values())
    products = sum(ms for name, ms in kernels.items()
                   if re.search('gemm|xmma|cutlass|matmul|sdpa', name, re.I))
    print(f'wall clock {step_ms:.2f} ms per step, device busy {busy:.2f} ms '
          f'({busy / step_ms:.0%}), matrix products {products / busy:.0%}')
    for name, ms in sorted(kernels.items(), key=lambda kv: -kv[1])[:rows]:
        name = re.sub(r'^void |\(anonymous namespace\)::|\w+::|\(.*', '', name)
        print(f'{ms:6.2f} ms {ms / busy:4.0%}  {name[:52]}')

We profile the training step:

In [ ]:
show_profile(kernel_times(step), 1000 * wall)

The device is busy for almost all of the wall-clock time, so the step is
not overhead-bound. Three quarters or more of the device time goes to a
handful of matrix-product kernels; their names contain *gemm*, the
traditional name for a general matrix product. These kernels are
compute-bound: a product of shapes $1024 \times 4096$ and
$4096 \times 4096$ has an intensity of about 340 FLOP/byte in fp32, against
a ridge point of about 82 for tf32 (82.6 TFLOP/s over 1.008 TB/s).

The remaining kernels are elementwise kernels and reductions: the ReLUs, the
bias gradients, which sum over the batch, and the SGD update. The update
reads every weight and its gradient and writes the weight back, 12 bytes
for two FLOPs. As a kernel of its own it is bandwidth-bound, and its time
does not depend on the batch size, so at small batch sizes it can take a
large share of the step.

The update is the `multi_tensor_apply_kernel` line, about a sixth of the
device time. It moves 12 bytes for each of the 34 million weights, 0.4 GB,
which takes 0.4 ms at the card's bandwidth.

The four steps of the method can now be stated with their tools.
**Measure** with `Benchmark` and a profiler. **Classify**: a busy device
dominated by matrix products is compute-bound; a busy device dominated by
elementwise and memory-bound kernels is bandwidth-bound; a device idle
between kernels is overhead-bound. **Fix** what limits the program: faster
formats and more devices for compute, fusion and recomputation for
bandwidth, larger calls and graph capture for overhead. **Re-measure**,
because each fix moves the bottleneck. We can now place any operation or
program in its regime. We cannot yet say where $P$ and $\beta$ come from, or
why the ridge point has risen over the hardware generations;
that section answers both questions.

## Summary

An operation on a GPU is limited by arithmetic, by memory bandwidth, or by
the fixed cost of issuing it from the host. Its arithmetic intensity, the
FLOPs it performs per byte it moves, decides between the first two through
the roofline bound $\min(P, I\beta)$. Below the ridge point $P/\beta$,
about 164 FLOP/byte on the RTX 4090, memory sets the time; the intensity of
a linear layer grows with the batch size. The third limit shows only in
measurement: an operation whose device time is shorter than the host's time
per call runs at the host's pace, so a $256 \times 256$ product reaches
less than two percent of the peak. Correct timing waits for the asynchronous
device before starting and before stopping the clock, keeps synchronization
points out of loops, and averages many calls, as `d2l.Benchmark` does. The
rest of the chapter applies one method, with a profiler to account for every
kernel of a program: measure, classify the regime, fix it, and measure
again.

## Exercises

1. **The smallest dimension.** The exact expression for the intensity of
   the product $\mathbf{X}\mathbf{W}$ in the equation, the
   one before the approximation, holds for all shapes, not only for
   $B \ll D, F$. Let $m = \min(B, D, F)$.
    1. Show that the intensity lies between $2m/(3b)$ and $2m/b$. Conclude
       that in bf16, on a GPU whose ridge point is 164 FLOP/byte, every
       matrix product with a dimension below 164 is bandwidth-bound in the
       roofline model.
    1. Apply the bound to three bf16 products: the final layer of the MLP
       of that section, which maps width 4096 to ten
       outputs, on a batch of 4096 examples; the attention scores
       $\mathbf{Q}\mathbf{K}^\top$ of one head with
       $\mathbf{Q}, \mathbf{K} \in \mathbb{R}^{T \times 64}$ for a context of
       $T$ tokens; and a $1 \times 1$ convolution (that section) from
       64 to 64 channels on a batch of 128 images of $32 \times 32$ pixels.
       For each product, name the dimensions that would have to grow for
       it to reach the ridge point, and state whether a practitioner
       controls them.
    1. A multi-head attention layer computes the scores of all its heads in
       one batched kernel. Does batching the heads change the intensity of
       the scores? State what your answer implies for an attention
       implementation that writes the $T \times T$ scores to memory.

1. [code] **Batch size and the ridge.** Consider the product
   $\mathbf{X}\mathbf{W}$ with $\mathbf{X} \in \mathbb{R}^{B \times 8192}$
   and $\mathbf{W} \in \mathbb{R}^{8192 \times 8192}$ in bf16.
    1. Using the equation without the approximation, find
       the batch size at which the product reaches the ridge point of your
       GPU. Compare it with the batch size that the approximation gives.
    1. Predict the roofline bound on the throughput for
       $B \in \{1, 8, 64, 512, 4096\}$. Measure the achieved TFLOP/s with
       `Benchmark`, and plot the bound and the measurements against $B$ on
       log-log axes. At $B = 1$, also convert the measured time into an
       achieved memory bandwidth, and compare it with the specification.
    1. When a language model generates text one token at a time, each of
       its linear layers runs a product of this kind with $B$ equal to the
       number of sequences served together. What does your plot imply about
       the time per step of serving 64 sequences instead of one?

3. [code] **Hidden barriers.** Each of the following lines runs once per
    step of a training loop, where `x` and `g` are tensors on the GPU and
    `c` is a Python float: `x[x > 0].sum()`, `print(x.shape)`,
    `x.argmax()`, `if g.norm() > c: g *= c / g.norm()`, and
    `x * torch.tensor(0.5, device=x.device)`.
    1. Decide which of the lines make the host wait for the device, and
       justify each decision.
    1. Rewrite every line that waits so that it computes the same result
       without a barrier.
    1. Call `torch.cuda.set_sync_debug_mode('warn')` before the loop, which
       makes PyTorch warn at every operation that synchronizes with the
       device, and check both answers.

4. **Diagnosis from timings.** Suppose that `Benchmark` reports the
   following times per call for four operations of a bf16 program on the
   RTX 4090 ($P = 165$ TFLOP/s, $\beta = 1.008$ TB/s). Operation A
   multiplies two $8192 \times 8192$ matrices in 7.4 ms. Operation B
   applies a softmax to the rows of an $8192 \times 8192$ matrix in
   0.30 ms; it reads and writes the matrix once and performs about five
   FLOPs per element. Operation C multiplies two $256 \times 256$ matrices
   in 10 µs. Operation D multiplies two $1024 \times 1024$ matrices in
   40 µs.
    1. For each operation, compute its arithmetic intensity and the
       fractions of $P$ and of $\beta$ that it achieves. Classify each
       operation twice: once by comparing its intensity with the ridge
       point, and once by the roof that it is close to. On which operations
       do the two classifications disagree?
    1. For each operation that is close to neither roof, name the most
       likely of the causes given in that section, and
       describe what a profiler timeline of the repeated calls would show
       that confirms it.

1. **Idle time in a profile.** Suppose that two programs keep the GPU busy
   for only a fifth of their wall-clock time. Program A trains a small MLP
   with many narrow layers and launches every operation from Python.
   Program B trains a large convolutional network, and its data loader
   decodes and resizes every image in the main Python process.
    1. Describe where the idle gaps of each program appear in a timeline.
       For each program, state whether one of the fixes of the method at the
       end of that section removes its gaps, and name that
       fix where one does.
    1. Describe an experiment that confirms the diagnosis of program B
       without a profiler.
    1. A third program keeps the GPU busy with large matrix products but
       converts its loss to a Python float after every step, and its
       profile attributes most of the host time to that conversion. Explain
       what the host and the device do during that time, and why that time
       is not overhead in the sense of the overhead-bound regime.

1. [code] **A pipeline model of dispatch.** ● Model one call of a GPU
   operation by three durations: the host needs $h$ to enqueue the call,
   the device needs $d$ to execute it, and a barrier adds a latency
   $\sigma$ after the queue drains. The device executes calls in the order
   in which they were enqueued, and it is idle when the first call is
   enqueued.
    1. Show that $R$ calls enqueued back to back and followed by one barrier
       take

        $$R \max(h, d) + \min(h, d) + \sigma.$$

       What does `Benchmark` therefore report for an operation with
       $d < h$? Explain why its result for `x + 1` on a $256 \times 256$
       tensor says nothing about the memory bandwidth of the kernel.
    1. Treat one step of the loops of that section as one
       call. The loop that converts the running total into a Python number
       on every step waits at a barrier after each step. Express the ratio
       of its time to the time of the loop that reads the total once in
       terms of $h$, $d$, and $\sigma$.
    1. Run both loops with `x` of side 256, 512, 1024, 2048, and 4096. From
       the ten timings, estimate $h$ and $\sigma$, which do not depend on
       the size, and $d$ for each size, and check that the model reproduces
       the measured ratios. Above which device time per step does reading a
       value on every step cost less than ten percent?

1. [code] **Tiles and waves.** ● Model a matrix-product kernel on a GPU
   with $S$ SMs as follows; the RTX 4090 has $S = 128$. The kernel splits the $n \times n$ output
   into $128 \times 128$ tiles. Each SM computes one tile at a time at
   $1/S$ of the peak, and the tiles run in waves of at most $S$.
    1. Derive the fraction of the peak that this model allows as a function
       of $n$ and $S$, and evaluate it with the $S$ of your GPU for the
       sizes of the sweep in that section. Compare the result
       with the fractions of the peak that your run of the sweep measures,
       and name the sizes at which the host time per call, not the tiles,
       limits the measurement.
    1. Predict the fraction for $n = 2176$, 2304, 2432, and 2560, all
       multiples of 128, relative to the fraction at $n = 2048$. Measure the
       five sizes with `Benchmark`, and explain where the measured pattern
       departs from the predicted one.
    1. Describe two ways in which a kernel can split the product at
       $n = 512$ into more pieces of work, so that all SMs take part, and
       state the cost that each way adds.

    *Adapted from NVIDIA, [Matrix Multiplication Background User's
    Guide](https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html),
    Section 3.2, Wave Quantization.*

1. [extended] **Roofline of a ResNet step.** ● Train `ResNet18` of
   that section on Fashion-MNIST at $96 \times 96$ pixels with batch
   size 128, as in that section, and map one training step onto the
   roofline of your GPU.
    1. For every convolution of the forward pass and for the final dense
       layer, compute the FLOPs and the bytes, counting the input, the
       weights, and the output once each. Do the same for one batch
       normalization and one ReLU of each stage. Mark every operation at its
       intensity on a roofline plot of your GPU.
    1. Profile one training step. Split its device time among convolutions,
       normalizations and activations, and all other kernels, add the idle
       time between kernels, and report the share of each in the
       wall-clock time of the step.
    1. Apply the fix of the method that targets the largest share outside
       the compute-bound regime, measure again, and report which share
       dominates afterward.

[Discussions](https://d2l.discourse.group/t/36787)